In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

In [2]:
chars = list("abcdefghijklmnopqrstuvwxyz")
num_chars = len(chars)

char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

max_len = 10
min_len = 5
latent_dim = 128
num_samples = 10000

In [3]:
def generate_data(num_samples):
    encoder_input_data = []
    decoder_input_data = []
    decoder_target_data = []
    original_texts = []

    for _ in range(num_samples):
        length = np.random.randint(min_len, max_len + 1)
        seq = ''.join(np.random.choice(chars, length))

        remove_idx = np.random.randint(len(seq))
        corrupted = seq[:remove_idx] + seq[remove_idx+1:]

        seq_padded = seq.ljust(max_len)
        corrupted_padded = corrupted.ljust(max_len)

        enc = np.zeros((max_len, num_chars))
        dec_in = np.zeros((max_len, num_chars))
        dec_out = np.zeros((max_len, num_chars))

        for t, char in enumerate(corrupted_padded):
            if char != ' ':
                enc[t, char_to_idx[char]] = 1

        for t, char in enumerate(seq_padded):
            if char != ' ':
                dec_in[t, char_to_idx[char]] = 1
                if t > 0:
                    dec_out[t-1, char_to_idx[char]] = 1

        encoder_input_data.append(enc)
        decoder_input_data.append(dec_in)
        decoder_target_data.append(dec_out)
        original_texts.append(seq)

    return (
        np.array(encoder_input_data),
        np.array(decoder_input_data),
        np.array(decoder_target_data),
        original_texts
    )

In [4]:
encoder_input_data, decoder_input_data, decoder_target_data, originals = generate_data(10000)

In [5]:
encoder_inputs = Input(shape=(None, num_chars))
encoder_lstm = LSTM(latent_dim, return_state=True)

_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

In [6]:
decoder_inputs = Input(shape=(None, num_chars))

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs,
    initial_state=encoder_states
)

decoder_dense = Dense(num_chars, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [7]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy'
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, None, 26)          │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_1 (InputLayer)    │ (None, None, 26)          │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ [(None, 128), (None,      │          79,360 │ input_layer[0][0]          │
│                               │ 128), (None, 128)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_1 (LSTM)                 │ [(None, None, 128),       │          79,360 │ input_layer_1[0][0],       │
│                               │ (None, 128), (None, 128)] │                 │ lstm[0][1], lstm[0][2]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, None, 26)          │           3,354 │ lstm_1[0][0]               │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 162,074 (633.10 KB)

 Trainable params: 162,074 (633.10 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=15,
    validation_split=0.2
)

Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - loss: 2.0821 - val_loss: 1.9523
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - loss: 1.8120 - val_loss: 1.7273
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - loss: 1.6418 - val_loss: 1.5857
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - loss: 1.5241 - val_loss: 1.4529
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - loss: 1.4201 - val_loss: 1.4081
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - loss: 1.3455 - val_loss: 1.3320
Epoch 7/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - loss: 1.2849 - val_loss: 1.2916
Epoch 8/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - loss: 1.2409 - val_loss: 1.2347
Epoch 9/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - loss: 1.2125 - val_loss: 1.2280
Epoch 10/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - loss: 1.1942 - val_loss: 1.2336
Epoch 11/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - loss: 1.1682 - val_loss: 1.2188
Epoch 12/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 

In [9]:
encoder_model = Model(encoder_inputs, encoder_states)

In [10]:
# wejścia dla stanów
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# decoder używa stanów jako wejścia
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

In [11]:
def decode_sequence(input_seq):
    # encoder daje stan
    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1, 1, num_chars))
    decoded_sentence = ""

    for _ in range(max_len):
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = idx_to_char[sampled_token_index]

        decoded_sentence += sampled_char

        # aktualizacja wejścia
        target_seq = np.zeros((1, 1, num_chars))
        target_seq[0, 0, sampled_token_index] = 1

        # aktualizacja stanów
        states_value = [h, c]

    return decoded_sentence

In [12]:
test_input = encoder_input_data[0:1]

print("Wynik:", decode_sequence(test_input))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Wynik: ohzdrgssss


In [13]:
i = 0

input_seq = encoder_input_data[i:i+1]

print("ORYGINAŁ:", originals[i])
print("PREDYKCJA:", decode_sequence(input_seq))

ORYGINAŁ: ohzdrcq
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
PREDYKCJA: ohzdrgssss
